<a href="https://colab.research.google.com/github/ardominguezm/golden-age-semantic-reconfiguration/blob/main/notebooks/paper1_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 1 — Golden Age Semantic Reconfiguration

**Current stage: Phase 13 — Null Models and Counterfactual Rewiring**

## Scientific target
Paper 1 asks whether the Renaissance→Baroque transition is merely gradual semantic drift or whether the poetic semantic system undergoes **structural reorganization of relations among concepts**.

Phase 12 revealed an uncertainty-aware trajectory of lexical turnover and persistent-concept rewiring, but the strongest movement also coincided with large poem-set turnover. Phase 13 therefore asks the counterfactual question that must be answered before any historical claim:

> **Is observed rewiring among persistent concepts larger than expected from finite sampling, poem turnover, and author composition when semantic content is detached from temporal position?**

The frozen main representation remains **within-line concept co-occurrence, support ≥1, PPMI**. Historiographic dates (1580/1605), literary-period labels and change-point models are not used in either null. The two null models and their simulation budget were frozen in `manuscript/phase13_null_model_protocol.md` before results.


In [ ]:
import sys, subprocess, hashlib, urllib.request, re, shutil, unicodedata, math
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from functools import lru_cache
from difflib import SequenceMatcher
import numpy as np, pandas as pd
import xml.etree.ElementTree as ET

SPACY_VERSION='3.8.7'; MODEL_NAME='es_core_news_sm'; MODEL_VERSION='3.8.0'
MODEL_URL='https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl'
MODEL_SHA256='e451a83d6df79b87e9eed0cb553f03e99e36a3bab18a7b79f0dcfd1fdf875e12'
wheel=Path('/content/es_core_news_sm-3.8.0-py3-none-any.whl')
if not wheel.exists() or hashlib.sha256(wheel.read_bytes()).hexdigest()!=MODEL_SHA256:
    urllib.request.urlretrieve(MODEL_URL,wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest()==MODEL_SHA256
subprocess.run([sys.executable,'-m','pip','install','-q',f'spacy=={SPACY_VERSION}',str(wheel)],check=True)
import spacy
assert spacy.__version__==SPACY_VERSION
nlp=spacy.load(MODEL_NAME,disable=['parser','ner'])
assert nlp.meta.get('version')==MODEL_VERSION
print('Environment ready:',f'spaCy {spacy.__version__}',f'| {MODEL_NAME} {nlp.meta.get("version")}')


In [ ]:
SOURCES={
 'navarro_tei':('https://github.com/bncolorado/CorpusSonetosSigloDeOro.git','092a5fe70a4065a4d84bfed288bffd3851348f9c'),
 'gongora_scholarly':('https://github.com/gongoradigital/gongoraobra.git','3beadeecc059a7cc48499dc2683bb378a2630978'),
}
ROOT=Path('/content/gasr_phase13_sources'); ROOT.mkdir(exist_ok=True)
def clone(name,url,commit):
    dst=ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    subprocess.run(['git','clone','--quiet',url,str(dst)],check=True)
    subprocess.run(['git','-C',str(dst),'checkout','--quiet',commit],check=True)
    got=subprocess.check_output(['git','-C',str(dst),'rev-parse','HEAD'],text=True).strip()
    assert got==commit,(name,got,commit)
    return dst
paths={k:clone(k,*v) for k,v in SOURCES.items()}
N=paths['navarro_tei']; G=paths['gongora_scholarly']; XML_ID='{http://www.w3.org/XML/1998/namespace}id'
def local(tag): return tag.split('}')[-1] if '}' in tag else tag
def el_text(el): return '' if el is None else ' '.join(' '.join(el.itertext()).split())
def norm(s):
    s=unicodedata.normalize('NFKD',str(s)); s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]','',s.lower())
def years_1580_1626(s):
    return sorted(set(int(x) for x in re.findall(r'(?<!\d)(1[56]\d{2})(?!\d)',str(s)) if 1580<=int(x)<=1626))

rows=[]
for fp in sorted(N.rglob('*.xml')):
    root=ET.parse(fp).getroot()
    lines=[el_text(x) for x in root.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    author=fp.parent.name; txt='\n'.join(lines)
    rows.append({'n_id':f'{author}::{fp.name}','author_dir':author,'n_lines':len(lines),'lines':lines,
                 'text_tei':txt,'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2]))})
n=pd.DataFrame(rows); assert len(n)==5078

primary_rows=[]
def add(pid,author,lo,hi,confidence,basis):
    primary_rows.append({'n_id':pid,'author_dir':author,'composition_min':int(lo),'composition_max':int(hi),
                         'temporal_confidence':confidence,'temporal_basis':basis})

# Reproduce the 58 Góngora links frozen in Phases 4/5.
groot=ET.parse(G/'gongora_obra-poetica.xml').getroot()
parent={child:par for par in groot.iter() for child in par}; grows=[]
for el in groot.iter():
    xid=el.attrib.get(XML_ID,'')
    if local(el.tag)!='div' or not xid.lower().startswith('poem'): continue
    lines=[el_text(x) for x in el.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    vals=[]; cur=el
    for _ in range(6):
        vals+=list(cur.attrib.values())
        if cur.text: vals.append(cur.text)
        for ch in list(cur):
            if local(ch.tag) in {'head','date','label'}: vals.append(el_text(ch))
            if ch.tail: vals.append(ch.tail)
        cur=parent.get(cur)
        if cur is None: break
    ys=sorted(set(y for v in vals for y in years_1580_1626(v))); txt='\n'.join(lines)
    grows.append({'g_id':xid,'n_lines':len(lines),'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),
                  'scholarly_year':ys[0] if len(ys)==1 else pd.NA,
                  'year_status':'unique' if len(ys)==1 else ('ambiguous' if len(ys)>1 else 'missing')})
g=pd.DataFrame(grows); g14=g[(g.n_lines==14)&g.signature.ne('')].copy(); g_by_id=g.set_index('g_id',drop=False)
ng=n[n.author_dir.eq('Gongora')].copy(); sig_to_gids=g14.groupby('signature').g_id.apply(list).to_dict()
links=[]
for r in ng.itertuples(index=False):
    exact_ids=sig_to_gids.get(r.signature,[])
    if len(exact_ids)==1: gid,score,method=exact_ids[0],1.0,'exact'
    else:
        best_gid,best_score=None,-1.0
        for gr in g14.itertuples(index=False):
            sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
            if sc>best_score: best_gid,best_score=gr.g_id,sc
        gid,score,method=best_gid,best_score,'fuzzy'
    links.append({'n_id':r.n_id,'g_id':gid,'method':method,'score':float(score),'preaccept':method=='exact' or score>=0.98})
glink=pd.DataFrame(links); pre=glink[glink.preaccept].copy()
collisions=set(pre.g_id.value_counts()[lambda s:s>1].index)
glink['accept_phase4']=glink.preaccept&~glink.g_id.isin(collisions)
acc=glink[glink.accept_phase4].merge(g[['g_id','scholarly_year','year_status']],on='g_id',how='left')
acc=acc[acc.year_status.eq('unique')&acc.scholarly_year.notna()].copy()
for r in acc.itertuples(index=False):
    add(r.n_id,'Gongora',r.scholarly_year,r.scholarly_year,'A' if r.method=='exact' else 'B',
        'scholarly_chronology_year_exact_link' if r.method=='exact' else 'scholarly_chronology_year_fuzzy_link')
phase4_nids=set(acc.n_id); phase4_gids=set(acc.g_id); unmatched=ng[~ng.n_id.isin(phase4_nids)].copy()
first2_index=g14.groupby('first2_signature').g_id.apply(list).to_dict(); recovered=[]
for r in unmatched.itertuples(index=False):
    ids2=first2_index.get(r.first2_signature,[])
    if len(ids2)!=1: continue
    gid=ids2[0]
    if gid in phase4_gids: continue
    gr=g_by_id.loc[gid]; score=SequenceMatcher(None,r.signature,gr.signature).ratio()
    if score>=0.95 and gr.year_status=='unique' and pd.notna(gr.scholarly_year):
        recovered.append((r.n_id,gid,score,int(gr.scholarly_year)))
rec=pd.DataFrame(recovered,columns=['n_id','g_id','score','year'])
dup=set(rec.g_id.value_counts()[lambda s:s>1].index) if len(rec) else set(); rec=rec[~rec.g_id.isin(dup)]
for r in rec.itertuples(index=False): add(r.n_id,'Gongora',r.year,r.year,'B','scholarly_chronology_year_variant_link')
assert sum(x['author_dir']=='Gongora' for x in primary_rows)==58

GAR={**{i:(1526,1532,'B','scholarly_phase_interval') for i in [1,2,3,4,6,26,27]},
     25:(1534,1535,'B','scholarly_interval'),33:(1535,1535,'A','historically_anchored_scholarly_year'),
     35:(1535,1535,'A','historically_anchored_scholarly_year'),
     **{i:(1533,1535,'B','revised_scholarly_interval') for i in [7,8,12,15,19,28,30,31]}}
for no,(lo,hi,conf,basis) in GAR.items():
    add(f'GarcilasoDeLaVega::GarcilasoDeLaVega_{no:02d}.xml','GarcilasoDeLaVega',lo,hi,conf,basis)
for no,lo,hi,conf,basis in [(30,1596,1596,'B','Cadiz_1596'),(13,1598,1598,'A','FelipeII_tomb_1598'),(31,1597,1598,'B','Herrera_death_epitaph')]:
    add(f'Cervantes::Cervantes_{no}.xml','Cervantes',lo,hi,conf,basis)
for no,lo,hi,basis in [(224,1574,1574,'Alameda_CarlosV'),(279,1578,1579,'Barahona_Granada'),
                       (276,1573,1574,'Bazan_Tunis'),(300,1580,1582,'Portugal_to_H'),(281,1578,1578,'DonJuan_de_Austria')]:
    add(f'FernandoDeHerrera::FernandoDeHerrera_{no}.xml','FernandoDeHerrera',lo,hi,'B',basis)
for no in [2,19,4,5]:
    add(f'PedroEspinosa::PedroEspinosa_{no}.xml','PedroEspinosa',1594,1596,'B','Espinosa_happiness_period_1594_1596')
for no,year,basis in [(131,1609,'Carrillo_sonnet_1609'),(69,1611,'Aminta_1611'),(70,1611,'Aminta_1611'),
                      (72,1611,'Aminta_1611'),(76,1611,'Aminta_1611'),(42,1610,'HenryIV_1610'),
                      (43,1610,'HenryIV_1610'),(45,1610,'HenryIV_1610'),(44,1624,'Osuna_1624')]:
    add(f'Quevedo::Quevedo_{no}.xml','Quevedo',year,year,'B',basis)

primary=pd.DataFrame(primary_rows).drop_duplicates('n_id').copy()
expected={'Gongora':58,'GarcilasoDeLaVega':18,'Quevedo':9,'FernandoDeHerrera':5,'PedroEspinosa':4,'Cervantes':3}
assert len(primary)==97 and primary.groupby('author_dir').size().to_dict()==expected
primary=primary.merge(n[['n_id','text_tei','lines','n_lines']],on='n_id',how='left',validate='one_to_one')
assert primary.text_tei.notna().all()
print('FROZEN PRIMARY CHRONOLOGY REPRODUCED:',len(primary),'poems |',primary.author_dir.nunique(),'authors')


In [ ]:
MAIN_POS={'NOUN','VERB','ADJ','ADV'}; MIN_LEMMA_LEN=2
line_records=[]
for r in primary.itertuples(index=False):
    for line_no,line in enumerate(r.lines,1): line_records.append((r.n_id,r.author_dir,line_no,line))
docs=list(nlp.pipe([x[3] for x in line_records],batch_size=128)); assert len(docs)==len(line_records)
token_rows=[]
for (pid,author,line_no,_),doc in zip(line_records,docs):
    for t in doc:
        lemma=unicodedata.normalize('NFC',str(t.lemma_)).strip().lower(); pos=t.pos_
        keep=bool(t.is_alpha and pos in MAIN_POS and len(lemma)>=MIN_LEMMA_LEN)
        token_rows.append({'n_id':pid,'author_dir':author,'line_no':line_no,'lemma':lemma,'pos':pos,
                           'is_main_content':keep,'concept':f'{lemma}::{pos}' if keep else pd.NA})
tokens=pd.DataFrame(token_rows); main_tok=tokens[tokens.is_main_content].copy()
concept_df=(main_tok[['n_id','concept','lemma','pos']].drop_duplicates(['n_id','concept'])
            .groupby(['concept','lemma','pos']).n_id.nunique().rename('poem_df').reset_index())
concept_tf=main_tok.groupby('concept').size().rename('token_frequency').reset_index()
vocab=concept_df.merge(concept_tf,on='concept',how='left')
MAIN_VOCAB=set(vocab.loc[vocab.poem_df.ge(2),'concept'])
assert len(tokens)==10439 and int(tokens.is_main_content.sum())==4358
assert len(MAIN_VOCAB)==668

poem_line_units={r.n_id:[set() for _ in range(r.n_lines)] for r in primary.itertuples(index=False)}
for (pid,line_no),grp in main_tok[main_tok.concept.isin(MAIN_VOCAB)].groupby(['n_id','line_no']):
    poem_line_units[pid][int(line_no)-1]=set(grp.concept)
assert set(poem_line_units)==set(primary.n_id)
print('Phase-10/11B semantic representation reproduced: 668 concepts | poetic-line contexts')


In [ ]:
# Frozen observed and null-design constants.
SEED=20260825; MC_DRAWS=1000
MAIN_WINDOWS=[(s,s+19) for s in range(1565,1606,5)]
TRANSITIONS=[(i+1,MAIN_WINDOWS[i],MAIN_WINDOWS[i+1]) for i in range(len(MAIN_WINDOWS)-1)]
MODES=('raw','author_balanced')
SELECTED_REPRESENTATION='B_line_support1'; MIN_PAIR_SUPPORT=1

NULL_SEED=20260826
NULL_CHRONOLOGY_DRAWS=100
NULL_REPS_PER_DRAW=20
NULL_TOTAL=NULL_CHRONOLOGY_DRAWS*NULL_REPS_PER_DRAW
NULL_DRAW_IDS=np.linspace(0,MC_DRAWS-1,NULL_CHRONOLOGY_DRAWS,dtype=int)
assert len(np.unique(NULL_DRAW_IDS))==NULL_CHRONOLOGY_DRAWS and NULL_TOTAL==2000

null_spec=pd.DataFrame([
    ('N1_local','author-overlap-preserving local reassignment',NULL_CHRONOLOGY_DRAWS,NULL_REPS_PER_DRAW,NULL_TOTAL),
    ('N2_global','within-author sampled-year permutation',NULL_CHRONOLOGY_DRAWS,NULL_REPS_PER_DRAW,NULL_TOTAL),
],columns=['null_model','definition','chronology_draws','replicates_per_draw','total_counterfactuals'])
print('PHASE 13 NULL DESIGN FROZEN BEFORE NULL RESULTS')
display(null_spec)
print('Null seed:',NULL_SEED,'| observed MC:',MC_DRAWS,'| null counterfactuals/model:',NULL_TOTAL)


In [ ]:
primary_idx=primary.set_index('n_id',drop=False)
pid_author=primary.set_index('n_id').author_dir.to_dict()

@lru_cache(maxsize=1024)
def build_state(ids_tuple,mode):
    ids=list(ids_tuple)
    if not ids: return {'active':set(),'edges':{},'n_poems':0}
    sub=primary_idx.loc[ids]; author_counts=Counter(sub.author_dir)
    node_mass=defaultdict(float); pair_mass=defaultdict(float); total_mass=0.0
    for pid in ids:
        units=poem_line_units[pid]; U=len(units)
        author=pid_author[pid]
        unit_weight=1.0 if mode=='raw' else 1.0/(author_counts[author]*U)
        total_mass+=U*unit_weight
        for concepts in units:
            cs=sorted(concepts)
            for u in cs: node_mass[u]+=unit_weight
            for u,v in combinations(cs,2): pair_mass[(u,v)]+=unit_weight
    active={u for u,c in node_mass.items() if c>0}; edges={}
    for (u,v),pm in pair_mass.items():
        pij=pm/total_mass; pi=node_mass[u]/total_mass; pj=node_mass[v]/total_mass
        if min(pij,pi,pj)<=0: continue
        ppmi=max(0.0,math.log2(pij/(pi*pj)))
        if ppmi>0: edges[(u,v)]=ppmi
    return {'active':active,'edges':edges,'n_poems':len(ids)}

def cosine_distance_sparse(a,b):
    keys=set(a)|set(b)
    if not keys: return np.nan
    va=np.fromiter((a.get(k,0.0) for k in keys),float)
    vb=np.fromiter((b.get(k,0.0) for k in keys),float)
    den=np.linalg.norm(va)*np.linalg.norm(vb)
    return float(1.0-np.dot(va,vb)/den) if den>0 else np.nan

def compare_states(s1,s2):
    V1,V2=s1['active'],s2['active']; union=V1|V2; persistent=V1&V2
    lexical=1-len(persistent)/len(union) if union else np.nan
    e1={e:w for e,w in s1['edges'].items() if e[0] in persistent and e[1] in persistent}
    e2={e:w for e,w in s2['edges'].items() if e[0] in persistent and e[1] in persistent}
    return lexical,cosine_distance_sparse(e1,e2),len(persistent)

def ids_for_window(years,start,end,ids_array):
    return tuple(sorted(ids_array[(years>=start)&(years<=end)].tolist()))

ids_array=primary.n_id.to_numpy(object)
lo=primary.composition_min.to_numpy(int); hi=primary.composition_max.to_numpy(int)
rng_dates=np.random.default_rng(SEED)
sampled=np.empty((MC_DRAWS,len(primary)),dtype=int)
for j,(a,b) in enumerate(zip(lo,hi)):
    sampled[:,j]=a if a==b else rng_dates.integers(a,b+1,size=MC_DRAWS)

def trajectory_from_years(years,draw_label,model_label,rep_label):
    sets=[ids_for_window(years,s,e,ids_array) for s,e in MAIN_WINDOWS]
    rows=[]
    for tid,left_win,right_win in TRANSITIONS:
        left,right=sets[tid-1],sets[tid]
        for mode in MODES:
            L,R,npersist=compare_states(build_state(left,mode),build_state(right,mode))
            rows.append({'draw':draw_label,'rep':rep_label,'null_model':model_label,
                         'transition_index':tid,'transition_center':1577+5*(tid-1),'mode':mode,
                         'lexical_turnover':L,'rewiring_cosine':R,'n_persistent_concepts':npersist})
    return rows

print('State/trajectory functions ready')


In [ ]:
# Reproduce the observed Phase-12 trajectory from all 1,000 chronology realizations.
obs_rows=[]
for m in range(MC_DRAWS):
    obs_rows.extend(trajectory_from_years(sampled[m],m,'observed',0))
    if (m+1)%200==0: print('observed chronology',m+1,'/',MC_DRAWS)
observed=pd.DataFrame(obs_rows)

def summarize(df,prefix=''):
    rows=[]
    for (tid,center,mode),g in df.groupby(['transition_index','transition_center','mode'],sort=True):
        r={'transition_index':tid,'transition_center':center,'mode':mode,'n':len(g)}
        for col in ['lexical_turnover','rewiring_cosine']:
            x=g[col].astype(float).dropna()
            for q,name in [(0.10,'q10'),(0.50,'median'),(0.90,'q90'),(0.95,'q95'),(0.99,'q99')]:
                r[f'{prefix}{col}_{name}']=float(x.quantile(q))
        rows.append(r)
    return pd.DataFrame(rows)

observed_summary=summarize(observed,'obs_')
# Regression checks from the already-executed Phase 12; these do not tune Phase 13.
r1592=observed_summary[(observed_summary.transition_center==1592)&(observed_summary['mode']=='raw')].iloc[0]
b1592=observed_summary[(observed_summary.transition_center==1592)&(observed_summary['mode']=='author_balanced')].iloc[0]
assert abs(r1592.obs_lexical_turnover_median-0.4057971014492754)<1e-9
assert abs(r1592.obs_rewiring_cosine_median-0.224870)<5e-6
assert abs(b1592.obs_rewiring_cosine_median-0.233746)<5e-6
print('Phase-12 observed trajectory reproduced exactly enough for regression checks')
display(observed_summary[['transition_index','transition_center','mode','obs_lexical_turnover_median','obs_rewiring_cosine_median']])


In [ ]:
# N1 PRIMARY NULL: within each observed adjacent-window local union, preserve
# author-specific left/right/shared counts exactly and randomize poem identities.
def local_reassign(left,right,rng):
    L,R=set(left),set(right); U=L|R
    outL=set(); outR=set()
    authors=sorted({pid_author[p] for p in U})
    for a in authors:
        pool=sorted(p for p in U if pid_author[p]==a)
        shared=sum((p in L and p in R) for p in pool)
        left_only=sum((p in L and p not in R) for p in pool)
        right_only=sum((p in R and p not in L) for p in pool)
        perm=list(rng.permutation(pool))
        S=set(perm[:shared])
        A=set(perm[shared:shared+left_only])
        B=set(perm[shared+left_only:shared+left_only+right_only])
        outL|=S|A; outR|=S|B
    assert len(outL)==len(L) and len(outR)==len(R) and len(outL&outR)==len(L&R)
    assert Counter(pid_author[p] for p in outL)==Counter(pid_author[p] for p in L)
    assert Counter(pid_author[p] for p in outR)==Counter(pid_author[p] for p in R)
    return tuple(sorted(outL)),tuple(sorted(outR))

ss=np.random.SeedSequence(NULL_SEED); child_local,child_global=ss.spawn(2)
rng_local=np.random.default_rng(child_local)
local_rows=[]
for di,m in enumerate(NULL_DRAW_IDS):
    yrs=sampled[m]
    sets=[ids_for_window(yrs,s,e,ids_array) for s,e in MAIN_WINDOWS]
    for rep in range(NULL_REPS_PER_DRAW):
        for tid,left_win,right_win in TRANSITIONS:
            left,right=sets[tid-1],sets[tid]
            nl,nr=local_reassign(left,right,rng_local)
            for mode in MODES:
                L,R,npersist=compare_states(build_state(nl,mode),build_state(nr,mode))
                local_rows.append({'draw':int(m),'rep':rep,'null_model':'N1_local',
                                   'transition_index':tid,'transition_center':1577+5*(tid-1),'mode':mode,
                                   'lexical_turnover':L,'rewiring_cosine':R,'n_persistent_concepts':npersist})
    if (di+1)%20==0: print('N1 chronology groups',di+1,'/',NULL_CHRONOLOGY_DRAWS)
local_null=pd.DataFrame(local_rows)
assert len(local_null)==NULL_TOTAL*len(TRANSITIONS)*len(MODES)
print('N1 complete:',len(local_null),'transition×mode null evaluations')


In [ ]:
# N2 SECONDARY NULL: preserve each author's sampled-year multiset but permute years
# among poems of that same author, generating coherent 9-window trajectories.
author_positions={a:np.flatnonzero(primary.author_dir.to_numpy()==a) for a in sorted(primary.author_dir.unique())}
rng_global=np.random.default_rng(child_global)
global_rows=[]; max_rows=[]
for di,m in enumerate(NULL_DRAW_IDS):
    base_years=sampled[m]
    for rep in range(NULL_REPS_PER_DRAW):
        py=base_years.copy()
        for a,idx in author_positions.items():
            py[idx]=rng_global.permutation(base_years[idx])
        rr=trajectory_from_years(py,int(m),'N2_global',rep)
        global_rows.extend(rr)
        tmp=pd.DataFrame(rr)
        for mode in MODES:
            x=tmp[tmp['mode'].eq(mode)].rewiring_cosine.astype(float)
            max_rows.append({'draw':int(m),'rep':rep,'mode':mode,'max_rewiring':float(x.max())})
    if (di+1)%20==0: print('N2 chronology groups',di+1,'/',NULL_CHRONOLOGY_DRAWS)
global_null=pd.DataFrame(global_rows); global_max=pd.DataFrame(max_rows)
assert len(global_null)==NULL_TOTAL*len(TRANSITIONS)*len(MODES)
assert len(global_max)==NULL_TOTAL*len(MODES)
print('N2 complete:',len(global_null),'transition×mode evaluations |',len(global_max),'max-stat values')


In [ ]:
local_summary=summarize(local_null,'local_')
global_summary=summarize(global_null,'global_')
tests=observed_summary.merge(local_summary,on=['transition_index','transition_center','mode'],suffixes=('','_local'))
tests=tests.merge(global_summary,on=['transition_index','transition_center','mode'],suffixes=('','_global'))

test_rows=[]
for r in tests.itertuples(index=False):
    key=(r.transition_index,r.mode)
    l=local_null[(local_null.transition_index==key[0])&local_null['mode'].eq(key[1])].rewiring_cosine.dropna().to_numpy(float)
    g=global_null[(global_null.transition_index==key[0])&global_null['mode'].eq(key[1])].rewiring_cosine.dropna().to_numpy(float)
    mx=global_max[global_max['mode'].eq(key[1])].max_rewiring.dropna().to_numpy(float)
    obs=float(r.obs_rewiring_cosine_median)
    test_rows.append({
        'transition_index':r.transition_index,'transition_center':r.transition_center,'mode':r.mode,
        'obs_lexical_turnover_median':r.obs_lexical_turnover_median,'obs_rewiring_median':obs,
        'local_null_median':r.local_rewiring_cosine_median,'local_null_q90':r.local_rewiring_cosine_q90,
        'local_null_q95':r.local_rewiring_cosine_q95,'local_null_q99':r.local_rewiring_cosine_q99,
        'local_excess':obs-r.local_rewiring_cosine_median,
        'local_null_percentile':float(np.mean(l<=obs)),
        'local_empirical_p':float((1+np.sum(l>=obs))/(1+len(l))),
        'global_null_median':r.global_rewiring_cosine_median,'global_null_q90':r.global_rewiring_cosine_q90,
        'global_null_q95':r.global_rewiring_cosine_q95,'global_null_q99':r.global_rewiring_cosine_q99,
        'global_excess':obs-r.global_rewiring_cosine_median,
        'global_null_percentile':float(np.mean(g<=obs)),
        'global_empirical_p':float((1+np.sum(g>=obs))/(1+len(g))),
        'global_max_fwer_p':float((1+np.sum(mx>=obs))/(1+len(mx))),
    })
null_tests=pd.DataFrame(test_rows).sort_values(['transition_index','mode'])

print('PHASE 13 OBSERVED vs COUNTERFACTUAL REWIRING — no historiographic overlay')
display(null_tests)
print('\nInterpretation discipline: effect sizes + empirical tails only; no period labels or change-point claim.')


In [ ]:
OUT=Path('/content/gasr_phase13_outputs'); OUT.mkdir(exist_ok=True)
observed.to_csv(OUT/'phase13_observed_mc_trajectory.csv',index=False)
observed_summary.to_csv(OUT/'phase13_observed_summary.csv',index=False)
local_null.to_csv(OUT/'phase13_local_null_draws.csv',index=False)
local_summary.to_csv(OUT/'phase13_local_null_summary.csv',index=False)
global_null.to_csv(OUT/'phase13_global_null_draws.csv',index=False)
global_summary.to_csv(OUT/'phase13_global_null_summary.csv',index=False)
global_max.to_csv(OUT/'phase13_global_maxstat.csv',index=False)
null_tests.to_csv(OUT/'phase13_observed_vs_null_tests.csv',index=False)
null_spec.to_csv(OUT/'phase13_frozen_null_specification.csv',index=False)

assert len(primary)==97 and len(MAIN_VOCAB)==668 and len(MAIN_WINDOWS)==9 and len(TRANSITIONS)==8
assert len(observed)==MC_DRAWS*len(TRANSITIONS)*len(MODES)
assert len(local_null)==NULL_TOTAL*len(TRANSITIONS)*len(MODES)
assert len(global_null)==NULL_TOTAL*len(TRANSITIONS)*len(MODES)
assert len(null_tests)==len(TRANSITIONS)*len(MODES)
assert np.isfinite(null_tests[['local_empirical_p','global_empirical_p','global_max_fwer_p']].to_numpy()).all()

print('\nPHASE 13 CHECKPOINT')
print('-------------------')
print('Primary chronology:',len(primary),'poems |',primary.author_dir.nunique(),'authors')
print('Main representation: B_line_support1 | poetic line | support >=1 | PPMI')
print('Observed trajectory:',MC_DRAWS,'chronology realizations |',len(TRANSITIONS),'transitions')
print('N1 local null:',NULL_TOTAL,'counterfactuals per transition/mode')
print('N2 global null:',NULL_TOTAL,'coherent counterfactual trajectories')
print('Max-statistic familywise null computed: TRUE')
print('1580/1605 used in null construction or tuning: FALSE')
print('Historiographic labels used in null construction or tuning: FALSE')
print('Change point computed: FALSE')
print('Historical Renaissance/Baroque claim made: FALSE')
print('Next phase: representation/text/window/author robustness before historiographic validation')
print('Outputs:',OUT)
